# Phase 5: Evaluation & Comparison
Evaluating the trained models on Hindi, Marathi, and Tamil.

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load dataset
df = pd.read_csv("../data/phase2/phase2_dataset.csv")
df = df.dropna(subset=['label'])
df['label'] = df['label'].astype(int)
df['text'] = "Question: " + df['question'] + " Context: " + df['context'] + " Answer: " + df['generated_answer']

# Load Model
model_path = "../models/muril_hallucination_final"
tokenizer = AutoTokenizer.from_pretrained(model_path)
model = AutoModelForSequenceClassification.from_pretrained(model_path)
model.eval()

if torch.cuda.is_available():
    model = model.to('cuda')


In [ ]:
def predict(texts):
    inputs = tokenizer(texts, padding=True, truncation=True, max_length=512, return_tensors="pt")
    if torch.cuda.is_available():
        inputs = {k: v.to('cuda') for k, v in inputs.items()}
    with torch.no_grad():
        outputs = model(**inputs)
    predictions = torch.argmax(outputs.logits, dim=-1)
    return predictions.cpu().numpy()

# Note: In a real scenario, evaluate on the test split, not the entire dataset.
# We predict in batches
batch_size = 16
predictions = []
for i in range(0, len(df), batch_size):
    batch_texts = df['text'].iloc[i:i+batch_size].tolist()
    preds = predict(batch_texts)
    predictions.extend(preds)

df['prediction'] = predictions


In [ ]:
print("Overall Classification Report:")
print(classification_report(df['label'], df['prediction'], target_names=['Faithful', 'Hallucinated']))

for lang in df['language'].unique():
    print(f"--- Language: {lang} ---")
    lang_df = df[df['language'] == lang]
    if len(lang_df) > 0:
        print(classification_report(lang_df['label'], lang_df['prediction'], target_names=['Faithful', 'Hallucinated']))

# Error analysis by injection strategy
print("--- Error Analysis by Injection Strategy ---")
error_df = df[df['label'] != df['prediction']]
strategy_counts = error_df['injection_strategy'].value_counts()
print(strategy_counts)
